# HDB Resale Flat Prices — ETL Pipeline

This notebook drives the ETL pipeline implemented under `src/`. It:
1. Extracts the relevant data.gov.sg source files programmatically
2. Combines them into a single master dataset (union of all attributes)
3. Profiles the master dataset
4. Validates `month`, `town`, `flat_type`, `flat_model`, `storey_range` using rules derived from the data's own statistical properties
5. Recomputes `remaining_lease` (assume 99-year lease, as of today), floored to years+months
6. Resolves duplicate composite keys (keep higher `resale_price`)
7. Flags anomalous resale prices (documented IQR heuristic)
8. Builds the `resale_identifier` transformation column
9. Hashes the identifier irreversibly (SHA-256) while preserving uniqueness
10. Writes the 5 required output groups: `raw`, `cleaned`, `failed`, `transformed`, `hashed`

> **Network note:** `data.gov.sg` requires outbound internet access. If you are
> running this notebook in a network-restricted environment, set
> `USE_SAMPLE_DATA = True` in the first code cell below to exercise the full
> pipeline against a small schema-accurate synthetic dataset
> (`tests/generate_sample_data.py`) instead. In an environment with normal
> internet access, set it to `False` to pull the real data.gov.sg files.


In [40]:
import sys
sys.path.append("..")  # allow `import config`, `from src import ...` when running from notebooks/

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

import config
from src import extract, profile, validate, clean, transform, hash_utils

USE_SAMPLE_DATA = False  # set False to pull the real data.gov.sg files (requires internet access)


## 1. Extraction

`src/extract.py` calls data.gov.sg's Open Data API (`poll-download` pattern) for each of the
three dataset ids configured in `config.DATASET_IDS`, downloads the raw CSVs to `data/raw/`
untouched, and concatenates them with an **outer join on columns** so the master dataset
contains every attribute present in *any* source file (per the Data Quality Requirement #1).

No rows/columns are manually deleted anywhere in this step.


In [41]:
if USE_SAMPLE_DATA:
    from tests.generate_sample_data import build_sample_master_dataset
    master = build_sample_master_dataset()
else:
    master = extract.extract(force_download=False)

master.to_csv(config.RAW_DIR / "master_raw_combined.csv", index=False)
print(master.shape)
master.head()


2026-07-12 11:24:44,105 [INFO] Discovered 5 dataset id(s) in collection 189: ['d_8b84c4ee58e3cfc0ece0d773c8ca6abc', 'd_43f493c6c50d54243cc1eab0df142d6a', 'd_2d5ff9ea31397b66239f245f57751537', 'd_ebc5ab87086db484f88045b47411ebc5', 'd_ea9ed51da2787afaf8e51f827c304208']
2026-07-12 11:24:44,393 [INFO] Raw file already present, skipping download: /Users/lokeshrakurthi/Desktop/lokesh codes/hdb_etl_test_lokesh/data/raw/Resale flat prices based on registration date from Jan-2017 onwards.csv
2026-07-12 11:24:44,710 [INFO] Raw file already present, skipping download: /Users/lokeshrakurthi/Desktop/lokesh codes/hdb_etl_test_lokesh/data/raw/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv
2026-07-12 11:24:44,960 [INFO] Raw file already present, skipping download: /Users/lokeshrakurthi/Desktop/lokesh codes/hdb_etl_test_lokesh/data/raw/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv
2026-07-12 11:24:45,333 [INFO] Raw file already present, skipping downlo

(981450, 12)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,__source_dataset_id
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,Improved,1979,61 years 04 months,232000,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,New Generation,1978,60 years 07 months,250000,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,262000,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,New Generation,1980,62 years 01 month,265000,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,265000,d_8b84c4ee58e3cfc0ece0d773c8ca6abc


## 2. Data Profiling

Rather than depend on a heavy external profiling framework, `src/profile.py` implements the
profiling primitives the validation rules actually need: per-column completeness/cardinality,
categorical frequency tables, and numeric distribution stats. These outputs are what
Section 3's validation thresholds are *derived from* — not hardcoded assumptions.


In [42]:
report = profile.build_profile_report(master)
report["column_summary"]


,column,dtype,non_null_count,null_pct,n_unique,sample_values
0,month,object,981450,0.00,439,"[2017-01, 2017-02, 2017-03, 2017-04, 2017-05]"
1,town,object,981450,0.00,27,"[ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT..."
2,flat_type,object,981450,0.00,8,"[2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE]"
3,block,object,981450,0.00,2772,"[406, 108, 602, 465, 601]"
4,street_name,object,981450,0.00,595,"[ANG MO KIO AVE 10, ANG MO KIO AVE 4, ANG MO K..."
5,storey_range,object,981450,0.00,25,"[10 TO 12, 01 TO 03, 04 TO 06, 07 TO 09, 13 TO..."
6,floor_area_sqm,object,981450,0.00,389,"[44, 67, 68, 73, 74]"
7,flat_model,object,981450,0.00,34,"[Improved, New Generation, DBSS, Standard, Apa..."
8,lease_commence_date,object,981450,0.00,57,"[1979, 1978, 1980, 1981, 1976]"
9,remaining_lease,object,272400,72.25,750,"[61 years 04 months, 60 years 07 months, 62 ye..."


In [43]:
report.get("freq_town")


,town,count
0,TAMPINES,84087
1,YISHUN,73642
2,JURONG WEST,70177
3,WOODLANDS,69474
4,BEDOK,69258
5,ANG MO KIO,54162
6,HOUGANG,53502
7,BUKIT BATOK,47477
8,CHOA CHU KANG,40639
9,SENGKANG,36574


In [44]:
report.get("freq_flat_model")


,flat_model,count
0,Model A,216335
1,Improved,181116
2,New Generation,116243
3,NEW GENERATION,78898
4,IMPROVED,73589
5,MODEL A,70381
6,Premium Apartment,52127
7,Simplified,36337
8,Apartment,27249
9,Standard,26474


## 3. Cleaning & Validation

Steps, in order:
1. **Type coercion** — numeric columns cast, string columns trimmed (`src/clean.coerce_types`)
2. **Remaining lease** — assume 99-year HDB lease; remaining lease = 99y − (today − lease_commence_date),
   floored to whole years + months (`src/clean.compute_remaining_lease`)
3. **Field validation** — `month`, `town`, `flat_type`, `flat_model`, `storey_range`
   (`src/validate.run_all_validations`) — see docstrings in `src/validate.py` for how each
   rule is derived from the dataset's own statistical properties (frequency tables, rare-category
   thresholds) rather than an arbitrarily hardcoded whitelist.
4. **Duplicate composite-key resolution** — composite key = all columns except `resale_price`;
   keep the higher price, discard the rest to the failed ledger (`src/clean.resolve_duplicate_keys`)
5. **Anomalous price flagging** — Tukey 1.5×IQR fence, computed **within** each
   (`town`, `flat_type`) group rather than globally, since price distributions differ hugely
   across towns/flat types (`src/clean.flag_anomalous_prices`). This flags but does not discard —
   see `docs/ASSUMPTIONS.md` for the full rationale.


In [45]:
typed = clean.coerce_types(master)
typed = clean.compute_remaining_lease(typed)
typed[["lease_commence_date", "remaining_lease_years", "remaining_lease_months", "remaining_lease"]].head()


,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease
0,1979,51,6,51 years 6 months
1,1978,50,6,50 years 6 months
2,1980,52,6,52 years 6 months
3,1980,52,6,52 years 6 months
4,1980,52,6,52 years 6 months


In [46]:
validated = validate.run_all_validations(typed)

valid_df = validated.loc[validated["is_valid"]].copy()
invalid_df = validated.loc[~validated["is_valid"]].copy()
invalid_df["failure_reason"] = invalid_df["validation_reason"]

print(f"Valid: {len(valid_df)}  |  Invalid: {len(invalid_df)}")
invalid_df[["month", "town", "flat_type", "storey_range", "failure_reason"]].head(10)


Valid: 978403  |  Invalid: 3047


,month,town,flat_type,storey_range,failure_reason
307,2017-01,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
308,2017-01,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
309,2017-01,CENTRAL AREA,4 ROOM,37 TO 39,flat_model is a statistical rare/likely-typo c...
310,2017-01,CENTRAL AREA,5 ROOM,49 TO 51,flat_model is a statistical rare/likely-typo c...
1439,2017-02,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
1440,2017-02,CENTRAL AREA,4 ROOM,07 TO 09,flat_model is a statistical rare/likely-typo c...
1441,2017-02,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
1442,2017-02,CENTRAL AREA,4 ROOM,10 TO 12,flat_model is a statistical rare/likely-typo c...
2598,2017-03,CENTRAL AREA,4 ROOM,28 TO 30,flat_model is a statistical rare/likely-typo c...
2658,2017-03,CENTRAL AREA,4 ROOM,49 TO 51,flat_model is a statistical rare/likely-typo c...


In [47]:
deduped_df, dup_failed_df = clean.resolve_duplicate_keys(valid_df)
print(f"Kept: {len(deduped_df)}  |  Discarded as duplicates: {len(dup_failed_df)}")
dup_failed_df.head()


2026-07-12 11:24:54,966 [INFO] Duplicate-key resolution: discarded 31037 of 978403 rows


Kept: 947366  |  Discarded as duplicates: 31037


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,__source_dataset_id,remaining_lease_years,remaining_lease_months,date_valid,town_valid,flat_type_valid,flat_model_valid,storey_range_valid,is_valid,validation_reason,failure_reason
204327,2025-12,CLEMENTI,5 ROOM,445A,CLEMENTI AVE 3,31 TO 33,113.0,Improved,2021,93 years 6 months,1480000.0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,93,6,True,True,True,True,True,True,,duplicate composite key - lower resale_price d...
226047,2026-03,CLEMENTI,5 ROOM,445A,CLEMENTI AVE 3,22 TO 24,113.0,Improved,2021,93 years 6 months,1460000.0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,93,6,True,True,True,True,True,True,,duplicate composite key - lower resale_price d...
204320,2025-11,CLEMENTI,5 ROOM,445A,CLEMENTI AVE 3,10 TO 12,113.0,Improved,2021,93 years 6 months,1380000.0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,93,6,True,True,True,True,True,True,,duplicate composite key - lower resale_price d...
175569,2024-04,BUKIT TIMAH,EXECUTIVE,2,TOH YI DR,10 TO 12,146.0,Maisonette,1988,60 years 6 months,1330000.0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,60,6,True,True,True,True,True,True,,duplicate composite key - lower resale_price d...
204148,2025-08,CLEMENTI,4 ROOM,445B,CLEMENTI AVE 3,37 TO 39,93.0,Model A,2021,93 years 6 months,1200000.0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,93,6,True,True,True,True,True,True,,duplicate composite key - lower resale_price d...


In [48]:
cleaned_df = clean.flag_anomalous_prices(deduped_df)
print(f"Flagged as price-anomalous: {cleaned_df['is_price_anomalous'].sum()} / {len(cleaned_df)}")

cleaned_df.to_csv(config.CLEANED_DIR / "cleaned_resale_prices.csv", index=False)
cleaned_df[["month", "town", "flat_type", "resale_price", "is_price_anomalous"]].sort_values(
    "is_price_anomalous", ascending=False).head(10)


/Users/lokeshrakurthi/Desktop/lokesh codes/hdb_etl_test_lokesh/notebooks/../src/clean.py:145: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flags = out.groupby(group_keys, group_keys=False).apply(_flag)
2026-07-12 11:24:55,235 [INFO] Flagged 18287 / 947366 rows as price anomalies (IQR x1.5, grouped by ['town', 'flat_type'])


Flagged as price-anomalous: 18287 / 947366


,month,town,flat_type,resale_price,is_price_anomalous
224622,2026-04,BUKIT MERAH,5 ROOM,1728000.0,True
189953,2024-02,SERANGOON,5 ROOM,772888.0,True
208664,2025-06,KALLANG/WHAMPOA,3 ROOM,772000.0,True
640748,2013-12,SERANGOON,5 ROOM,772000.0,True
227813,2026-03,JURONG WEST,4 ROOM,772000.0,True
222972,2026-02,BEDOK,4 ROOM,772000.0,True
170052,2024-10,ANG MO KIO,4 ROOM,772000.0,True
180195,2024-09,JURONG EAST,5 ROOM,772277.0,True
117466,2022-09,ANG MO KIO,4 ROOM,772500.0,True
591753,2011-07,CENTRAL AREA,2 ROOM,262000.0,True


## 4. Data Transformation — `resale_identifier`

Composition (per assignment spec):
- `S` + first-3-digits-of-block (zero-padded) + first-2-digits-of-(avg resale price, grouped
  by year-month/town/flat_type) + month (2 digits) + first letter of town

Then: re-check for any duplicate records introduced by this step (keep higher price), and
finally hash the identifier irreversibly.


In [49]:
transformed_df = transform.build_resale_identifier(cleaned_df)
transformed_df, transform_dup_failed = transform.resolve_duplicate_transformed_rows(transformed_df)

transformed_df.to_csv(config.TRANSFORMED_DIR / "transformed_resale_prices.csv", index=False)
transformed_df[["block", "month", "town", "flat_type", "avg_price_group", "resale_identifier"]].head(10)


,block,month,town,flat_type,avg_price_group,resale_identifier
224622,96A,2026-04,BUKIT MERAH,5 ROOM,1.055782e+06,S0961004B
224594,9A,2026-03,BUKIT MERAH,5 ROOM,1.091486e+06,S0091003B
199476,275A,2025-11,BISHAN,5 ROOM,1.150127e+06,S2751111B
199498,135,2025-11,BISHAN,EXECUTIVE,1.284000e+06,S1351211B
218476,138A,2025-01,TOA PAYOH,5 ROOM,1.117713e+06,S1381101T
199503,194,2025-07,BISHAN,EXECUTIVE,1.344000e+06,S1941307B
174172,96A,2024-06,BUKIT MERAH,5 ROOM,1.128667e+06,S0961106B
174171,9B,2024-06,BUKIT MERAH,5 ROOM,1.128667e+06,S0091106B
233354,139B,2026-06,TOA PAYOH,5 ROOM,1.213611e+06,S1391206T
174447,126A,2024-09,BUKIT MERAH,5 ROOM,1.059765e+06,S1261009B


## 5. Hashing

SHA-256 is used — see `src/hash_utils.py` module docstring for the full rationale
(irreversibility, negligible collision probability at this dataset's scale, no dependency
required beyond the standard library). A uniqueness sanity check runs automatically and raises
if the hash ever collides two previously-distinct identifiers.


In [50]:
hashed_df = hash_utils.hash_identifier_column(transformed_df)
hashed_df.to_csv(config.HASHED_DIR / "hashed_resale_prices.csv", index=False)
hashed_df[["resale_identifier", "resale_identifier_hashed"]].head(10)


,resale_identifier,resale_identifier_hashed
224622,S0961004B,acf02627fe56dc5b7fb5dd8656b2165840840e43fcfa6d...
224594,S0091003B,bd82d896c8e514d3312fa1c9d46c1d1bfe7a2bd640125b...
199476,S2751111B,24ff15abe530361fcc5cef656eb1a34bf07a5b3517612a...
199498,S1351211B,e97eb6a66957414ab138bb44c6b61f8e1f9d356a854011...
218476,S1381101T,13c1fe55ac62d065f473d052bdc85ec5df64db787740d6...
199503,S1941307B,310d283f30f28c6c9d82a4baf0eadab7156de6793c1f27...
174172,S0961106B,e69766469a268070b50769d4b4a8b3a9c836dc442f85af...
174171,S0091106B,6566f1cd79839f79d6bf871c3832c92eeb2f610b0cbe78...
233354,S1391206T,23786bbec0321e79f5424751010a259d6747f80c4dbc11...
174447,S1261009B,8679dda28f51edbfc661d22a3d6daa2ae00cb9b28790f2...


## 6. Failed-Records Ledger

Consolidates every record rejected at any stage (validation failures, duplicate keys) with a
`failure_reason` column, for the Data Science team / QA to review.


In [51]:
all_failed = pd.concat([invalid_df, dup_failed_df, transform_dup_failed], axis=0,
                        ignore_index=True, sort=False)
all_failed.to_csv(config.FAILED_DIR / "failed_records.csv", index=False)

print(f"Total failed records: {len(all_failed)}")
all_failed[["month", "town", "flat_type", "storey_range", "failure_reason"]].head(15)


Total failed records: 34084


,month,town,flat_type,storey_range,failure_reason
0,2017-01,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
1,2017-01,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
2,2017-01,CENTRAL AREA,4 ROOM,37 TO 39,flat_model is a statistical rare/likely-typo c...
3,2017-01,CENTRAL AREA,5 ROOM,49 TO 51,flat_model is a statistical rare/likely-typo c...
4,2017-02,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
5,2017-02,CENTRAL AREA,4 ROOM,07 TO 09,flat_model is a statistical rare/likely-typo c...
6,2017-02,CENTRAL AREA,4 ROOM,04 TO 06,flat_model is a statistical rare/likely-typo c...
7,2017-02,CENTRAL AREA,4 ROOM,10 TO 12,flat_model is a statistical rare/likely-typo c...
8,2017-03,CENTRAL AREA,4 ROOM,28 TO 30,flat_model is a statistical rare/likely-typo c...
9,2017-03,CENTRAL AREA,4 ROOM,49 TO 51,flat_model is a statistical rare/likely-typo c...


## 7. Summary

| Stage        | Row count |
|--------------|-----------|
| Raw (master) | see below |
| Cleaned      | see below |
| Transformed  | see below |
| Hashed       | see below |
| Failed       | see below |


In [52]:
print(f"raw={len(master)}  cleaned={len(cleaned_df)}  transformed={len(transformed_df)}  "
      f"hashed={len(hashed_df)}  failed={len(all_failed)}")


raw=981450  cleaned=947366  transformed=947366  hashed=947366  failed=34084


## Appendix: Running the pipeline as a script

Everything above is also encapsulated in `run_pipeline.py` for non-interactive / CI use:

```bash
python run_pipeline.py --use-sample-data      # sandbox / offline demo
python run_pipeline.py                        # real data.gov.sg pull
python run_pipeline.py --force-download       # bypass the local raw/ cache
```
